# Notebook 03b: Experimentación de Clasificación Multietiqueta (Multilabel)

## Contexto y Objetivo
Estudio exploratorio para predecir simultáneamente múltiples atributos de un siniestro automotor (ej. presencia de daño, tipo de impacto, zona afectada) en una única pasada forward de la red.

Se analiza la transición de una activación `Softmax` (excluyente) a `Sigmoid` (independiente) y el uso de `BinaryCrossentropy` por etiqueta.


In [1]:
#LIBRERIAS
import os
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import numpy as np
import keras
from keras import layers
import matplotlib.pyplot as plt
import pandas as pd
import cv2 as cv
import config
from IPython.display import Image
from tensorflow.keras.preprocessing import image
from tensorflow.keras.optimizers import Adam
import matplotlib.image as mpimg

In [2]:
processed_path = config.PROCESSED_PATH_ORIGINAL
df_train = pd.read_csv(F"{processed_path}/csv/train.csv")
df_test = pd.read_csv(F"{processed_path}/csv/test.csv")

In [3]:
x_train_csv =[]
for img in df_train['image_path']:
    imgPaths = cv.imread(img)
    x_train_csv.append(cv.resize(imgPaths,(180,180)))
y_train_csv = df_train['damaged']

x_test_csv = []
for img in df_test['image_path']:
    imgPaths = cv.imread(img)
    x_test_csv.append(cv.resize(imgPaths, (180, 180)))
y_test_csv = df_test['damaged']

y_train_encoded = to_categorical(y_train_csv)
y_test_encoded = to_categorical(y_test_csv)
X_train = tf.convert_to_tensor(x_train_csv)
X_test = tf.convert_to_tensor(x_test_csv)
y_train = tf.convert_to_tensor(y_train_encoded)
y_test = tf.convert_to_tensor(y_test_encoded)
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(32)
val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(32)

In [4]:
callbacks_list = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        mode='auto'
    ),
    keras.callbacks.ModelCheckpoint(
        './poto/reg_model_{val_loss:.3f}.keras',
        save_best_only = True,
        save_weights_only=False,
        monitor='val_loss'
    )
]

In [5]:
dnn_model = keras.Sequential()
imported_model = keras.applications.ResNet50(
    include_top=False,
    input_shape=(180,180,3),
    pooling='avg',
    classes=3,
    weights='imagenet'
)
for layer in imported_model.layers:
    layer.trainable=False
dnn_model.add(imported_model)
dnn_model.add(layers.Flatten())
dnn_model.add(layers.Dense(512, activation='relu'))
dnn_model.add(layers.Dense(3, activation='softmax'))
dnn_model.compile(optimizer=Adam(learning_rate=0.001),loss='categorical_crossentropy',metrics=['accuracy'])

In [6]:
history = dnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50
)

Epoch 1/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 76s 847ms/step - accuracy: 0.8248 - loss: 1.9439 - val_accuracy: 0.4068 - val_loss: 11.0404
Epoch 2/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 74s 888ms/step - accuracy: 0.5381 - loss: 5.4853 - val_accuracy: 0.4068 - val_loss: 7.1363
Epoch 3/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 75s 901ms/step - accuracy: 0.4831 - loss: 4.1587 - val_accuracy: 0.4068 - val_loss: 5.4199
Epoch 4/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 59s 711ms/step - accuracy: 0.6030 - loss: 2.8790 - val_accuracy: 0.4068 - val_loss: 3.6796
Epoch 5/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 66s 801ms/step - accuracy: 0.4857 - loss: 1.9574 - val_accuracy: 0.4280 - val_loss: 1.7116
Epoch 6/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 64s 777ms/step - accuracy: 0.6270 - loss: 0.9973 - val_accuracy: 0.4280 - val_loss: 2.0083
Epoch 7/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 62s 742ms/step - accuracy: 0.6448 - loss: 1.1774 - val_accuracy: 0.4280 - val_loss: 3.3375
Epoch 8/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 55s 668ms/step - accuracy: 0.6122 - loss: 1.7401 - val_acc

In [7]:
dnn_model.save("./models/modelo_prueba_more_data_multilab.keras")

In [8]:
Validation_Path = "./data/processed/original/csv/val.csv"
df_val = pd.read_csv(Validation_Path)
predictions = []

In [9]:
Classes = ["No Damage", "Damaged", "Unknown"]

In [10]:
for index, row in df_val.iterrows():
    Image_Keras = keras.utils.load_img(row['image_path'], target_size=(180,180))
    Image_Keras = keras.utils.img_to_array(Image_Keras)
    Image_Keras = keras.ops.expand_dims(Image_Keras, 0)
    predict = dnn_model.predict(Image_Keras)
    predicted_class=Classes[np.argmax(predict)]
    predictions.append(predicted_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step


IndexError: list index out of range